# KB Injection Reproduction Evaluation: Partial vs Full Match

This notebook implements the stricter evaluation metric your supervisor suggested for the 50 curated problems.

The paper currently reports a lenient metric: a problem counts as correct if **at least one** generated KB injection matches the reference KB for that problem. Here we keep that metric and add two stricter variants:

- **partial_match**: at least one generated relation is also in the gold/reference set.
- **full_match**: every required gold relation appears in the generated set.
- **exact_set_match**: the generated set is exactly the gold set, with no missing and no extra relations.

For the paper, I would report **partial_match** and **full_match**. The exact version is included as a useful diagnostic.

## Expected Input Format

Create a CSV file with one row per model output per problem. The default path below is:

`data/model_kb_evaluation.csv`

Required columns:

- `problem_id`: ID of the NLI problem.
- `dataset`: `SICK` or `SNLI`.
- `model`: model name, e.g. `gpt-oss-20b`.
- `gold_kb`: reference KB relations, separated by newlines, semicolons, or pipes.
- `generated_kb`: generated KB relations, separated by newlines, semicolons, or pipes.

Example values:

`isa(woman, person); isa(handle, hold)`

`isa(woman, person)`

In [2]:
from __future__ import annotations

import csv
import re
from collections import defaultdict
from pathlib import Path
from statistics import mean
from typing import Dict, Iterable, List, Set, Tuple

## Configuration

Change `DATA_PATH` if your curated 50-problem table lives somewhere else.

In [3]:
DATA_PATH = Path("data/model_kb_evaluation.csv")

# If True, normalize `isa_wn(...)` to `isa(...)`, since the codebase and paper use both names.
NORMALIZE_ISA_WN_TO_ISA = True

# If True, normalize KB argument direction by sorting arguments.
# Keep this False for the main paper metric, because direction matters for isa relations.
# You can set it True for an exploratory direction-insensitive diagnostic.
DIRECTION_INSENSITIVE = False

## Relation Parsing and Normalization

This parser makes the metric robust to whitespace, duplicate rows, and separator differences. It intentionally does **not** do semantic matching: `isa(handle, hold)` and `isa(grab, hold)` remain different.

In [4]:
RELATION_RE = re.compile(r"^\s*([A-Za-z_][A-Za-z0-9_]*)\s*\(\s*(.*?)\s*,\s*(.*?)\s*\)\s*$")


def split_kb_cell(value: object) -> List[str]:
    """Split a CSV cell into individual KB relation strings."""
    if value is None:
        return []
    text = str(value).strip()
    if not text:
        return []

    # Accept common separators used in spreadsheets and hand-written tables.
    parts = re.split(r"[\n;|]+", text)
    return [part.strip() for part in parts if part.strip()]


def normalize_arg(arg: str) -> str:
    """Normalize one relation argument without changing its meaning."""
    arg = arg.strip().lower()
    arg = arg.replace("_", " ")
    arg = re.sub(r"\s+", " ", arg)
    return arg


def normalize_relation(relation: str, *, direction_insensitive: bool = False) -> str:
    """Return canonical `predicate(arg1, arg2)` format for exact set comparison."""
    match = RELATION_RE.match(relation)
    if not match:
        raise ValueError(f"Invalid KB relation format: {relation!r}")

    pred, arg1, arg2 = match.groups()
    pred = pred.lower().strip()
    if NORMALIZE_ISA_WN_TO_ISA and pred == "isa_wn":
        pred = "isa"

    arg1 = normalize_arg(arg1)
    arg2 = normalize_arg(arg2)

    if direction_insensitive:
        arg1, arg2 = sorted([arg1, arg2])

    return f"{pred}({arg1}, {arg2})"


def normalize_kb_set(value: object, *, direction_insensitive: bool = False) -> Set[str]:
    """Parse and normalize a KB cell into a set of unique relations."""
    return {
        normalize_relation(rel, direction_insensitive=direction_insensitive)
        for rel in split_kb_cell(value)
    }

## Metric Functions

The central distinction is between overlap and complete recovery of the reference set.

In [5]:
def score_kb(gold: Set[str], generated: Set[str]) -> Dict[str, object]:
    """Score one problem/model output against the gold KB set."""
    matched = gold & generated
    missing = gold - generated
    extra = generated - gold

    return {
        "gold_count": len(gold),
        "generated_count": len(generated),
        "matched_count": len(matched),
        "partial_match": bool(matched),
        "full_match": bool(gold) and gold.issubset(generated),
        "exact_set_match": bool(gold) and gold == generated,
        "relation_precision": len(matched) / len(generated) if generated else 0.0,
        "relation_recall": len(matched) / len(gold) if gold else 0.0,
        "matched_relations": sorted(matched),
        "missing_relations": sorted(missing),
        "extra_relations": sorted(extra),
    }


def pct(values: Iterable[bool]) -> float:
    values = list(values)
    return 100 * sum(values) / len(values) if values else 0.0

## Load Data

If `data/model_kb_evaluation.csv` does not exist yet, the notebook uses a tiny example dataset so you can see the metric behavior immediately.

In [6]:
example_rows = [
    {
        "problem_id": "sick_example_full",
        "dataset": "SICK",
        "model": "example-model",
        "gold_kb": "isa(woman, person); isa(handle, hold)",
        "generated_kb": "isa(woman, person); isa(handle, hold)",
    },
    {
        "problem_id": "sick_example_partial",
        "dataset": "SICK",
        "model": "example-model",
        "gold_kb": "isa(woman, person); isa(handle, hold)",
        "generated_kb": "isa(woman, person)",
    },
    {
        "problem_id": "snli_example_extra",
        "dataset": "SNLI",
        "model": "example-model",
        "gold_kb": "isa(talk, speak)",
        "generated_kb": "isa(talk, speak); isa(person, human)",
    },
    {
        "problem_id": "snli_example_none",
        "dataset": "SNLI",
        "model": "example-model",
        "gold_kb": "isa(pull, drag)",
        "generated_kb": "isa(rope, object)",
    },
]


def load_rows(path: Path) -> List[Dict[str, str]]:
    if not path.exists():
        print(f"{path} not found; using built-in example rows.")
        return example_rows

    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


rows = load_rows(DATA_PATH)
len(rows), rows[:2]

data/model_kb_evaluation.csv not found; using built-in example rows.


(4,
 [{'problem_id': 'sick_example_full',
   'dataset': 'SICK',
   'model': 'example-model',
   'gold_kb': 'isa(woman, person); isa(handle, hold)',
   'generated_kb': 'isa(woman, person); isa(handle, hold)'},
  {'problem_id': 'sick_example_partial',
   'dataset': 'SICK',
   'model': 'example-model',
   'gold_kb': 'isa(woman, person); isa(handle, hold)',
   'generated_kb': 'isa(woman, person)'}])

## Score Every Row

In [7]:
scored_rows = []

for row in rows:
    gold = normalize_kb_set(row.get("gold_kb"), direction_insensitive=DIRECTION_INSENSITIVE)
    generated = normalize_kb_set(row.get("generated_kb"), direction_insensitive=DIRECTION_INSENSITIVE)
    scores = score_kb(gold, generated)

    scored_rows.append({
        **row,
        **scores,
        "gold_normalized": sorted(gold),
        "generated_normalized": sorted(generated),
    })

scored_rows[:2]

[{'problem_id': 'sick_example_full',
  'dataset': 'SICK',
  'model': 'example-model',
  'gold_kb': 'isa(woman, person); isa(handle, hold)',
  'generated_kb': 'isa(woman, person); isa(handle, hold)',
  'gold_count': 2,
  'generated_count': 2,
  'matched_count': 2,
  'partial_match': True,
  'full_match': True,
  'exact_set_match': True,
  'relation_precision': 1.0,
  'relation_recall': 1.0,
  'matched_relations': ['isa(handle, hold)', 'isa(woman, person)'],
  'missing_relations': [],
  'extra_relations': [],
  'gold_normalized': ['isa(handle, hold)', 'isa(woman, person)'],
  'generated_normalized': ['isa(handle, hold)', 'isa(woman, person)']},
 {'problem_id': 'sick_example_partial',
  'dataset': 'SICK',
  'model': 'example-model',
  'gold_kb': 'isa(woman, person); isa(handle, hold)',
  'generated_kb': 'isa(woman, person)',
  'gold_count': 2,
  'generated_count': 1,
  'matched_count': 1,
  'partial_match': True,
  'full_match': False,
  'exact_set_match': False,
  'relation_precision': 1

## Summary Tables

This reports the model-level numbers you can put in the paper. With 50 problems, each problem corresponds to 2 percentage points.

In [ ]:
def summarize(group_rows: List[Dict[str, object]]) -> Dict[str, object]:
    n = len(group_rows)
    return {
        "n": n,
        "partial_match_n": sum(bool(r["partial_match"]) for r in group_rows),
        "partial_match_pct": pct(r["partial_match"] for r in group_rows),
        "full_match_n": sum(bool(r["full_match"]) for r in group_rows),
        "full_match_pct": pct(r["full_match"] for r in group_rows),
        "exact_set_match_n": sum(bool(r["exact_set_match"]) for r in group_rows),
        "exact_set_match_pct": pct(r["exact_set_match"] for r in group_rows),
        "mean_relation_precision": mean(float(r["relation_precision"]) for r in group_rows) if n else 0.0,
        "mean_relation_recall": mean(float(r["relation_recall"]) for r in group_rows) if n else 0.0,
    }


def grouped_summary(rows: List[Dict[str, object]], keys: Tuple[str, ...]) -> List[Dict[str, object]]:
    groups = defaultdict(list)
    for row in rows:
        groups[tuple(row.get(k, "") for k in keys)].append(row)

    summary_rows = []
    for key_values, group_rows in sorted(groups.items()):
        summary_rows.append({**dict(zip(keys, key_values)), **summarize(group_rows)})
    return summary_rows


overall_by_model = grouped_summary(scored_rows, ("model",))
by_model_and_dataset = grouped_summary(scored_rows, ("model", "dataset"))

overall_by_model

In [ ]:
by_model_and_dataset

## Inspect Cases Where Partial Match Succeeds but Full Match Fails

These are the important cases for your supervisor's concern: the model found something useful, but did not recover the complete reference KB set.

In [ ]:
partial_not_full = [
    row for row in scored_rows
    if row["partial_match"] and not row["full_match"]
]

for row in partial_not_full:
    print(f"{row.get('model')} | {row.get('dataset')} | {row.get('problem_id')}")
    print(f"  matched: {row['matched_relations']}")
    print(f"  missing: {row['missing_relations']}")
    print(f"  extra:   {row['extra_relations']}")
    print()

## Inspect Full Match Cases with Extra Relations

These pass `full_match` because all gold relations are present, but fail `exact_set_match` because the model generated additional relations. Decide with your supervisor whether these should count as complete recovery or be penalized.

In [ ]:
full_with_extra = [
    row for row in scored_rows
    if row["full_match"] and not row["exact_set_match"]
]

for row in full_with_extra:
    print(f"{row.get('model')} | {row.get('dataset')} | {row.get('problem_id')}")
    print(f"  gold:  {row['gold_normalized']}")
    print(f"  extra: {row['extra_relations']}")
    print()

## Save Scored Rows and Summaries

The scored file is useful for error analysis. The summary files can be copied into the results table.

In [ ]:
OUTPUT_DIR = Path("evaluation_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def stringify_for_csv(value: object) -> object:
    if isinstance(value, list):
        return " | ".join(str(v) for v in value)
    return value


def write_csv(path: Path, rows: List[Dict[str, object]]) -> None:
    if not rows:
        return
    fieldnames = list(rows[0].keys())
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: stringify_for_csv(v) for k, v in row.items()})


write_csv(OUTPUT_DIR / "kb_reproduction_scored_rows.csv", scored_rows)
write_csv(OUTPUT_DIR / "kb_reproduction_summary_by_model.csv", overall_by_model)
write_csv(OUTPUT_DIR / "kb_reproduction_summary_by_model_and_dataset.csv", by_model_and_dataset)

print(f"Wrote outputs to {OUTPUT_DIR.resolve()}")

## Suggested Wording for the Paper

> We report both partial match accuracy and full match accuracy for KB reproduction. Partial match accuracy counts a problem as correct if at least one generated injection matches a reference injection for that problem. Full match accuracy counts a problem as correct only if the generated KB contains all reference injections for that problem. Because the reference KBs are manually validated minimal KB sets, full match accuracy measures whether the model recovers the complete minimal explanation rather than merely one useful relation.

If you choose to penalize additional generated relations, replace the last sentence with:

> We additionally report exact set match accuracy, which requires the generated KB set to be identical to the reference set.